# Wind Speed Histogram Workflow

This notebook walks through the `WindSpeedHistogram` workflow as *explicit, auditable notebook steps* instead of hiding the POST and GET calls behind a demo helper.

**What this notebook covers**

- **Import** the required SDK components and helper utilities.
- **Configure** the workbook path, API root, project site, token source, and optional analysis identifier.
- **Prepare** metadata and workbook rows before upload.
- **Upload** results only when a backend analysis does not already exist.
- **Retrieve** persisted results from the API and reconstruct the normalized analysis frame.
- **Plot** the retrieved results through `ResultsService`.

**How to use it**

- Run the notebook *top to bottom* the first time.
- Set `ANALYSIS_ID` when you want to *reuse* an existing backend analysis and skip the POST step.
- Review the Mermaid diagrams before each stage to understand *inputs, transformations, and outputs*.

## Mermaid Color Legend

The workflow diagrams in this notebook use a consistent visual language.

- <span style="color:#2B6F77;"><strong>Blue nodes</strong></span>: API calls, services, or external system interactions.
- <span style="color:#5E8C61;"><strong>Green nodes</strong></span>: validated outputs, identifiers, or reconstructed result frames.
- <span style="color:#C08B3E;"><strong>Amber nodes</strong></span>: transformation, serialization, or intermediate processing steps.
- <span style="color:#C47A2C;"><strong>Orange decision/request nodes</strong></span>: branching logic, runtime choices, or request definitions.
- <span style="color:#365A73;"><strong>Blue-grey connectors</strong></span>: data flow between notebook steps.

*Read the diagrams from top to bottom to follow the lifecycle from raw inputs to plotted outputs.*

## Imports

These imports assemble the *public SDK surface* used throughout the workflow.

**This step prepares**

- **Workbook access** with `pathlib`, `pandas`, and `re` for bin-edge parsing.
- **Metadata lookups** through `GeometryAPI` and `LocationsAPI`.
- **Analysis and API operations** through `ResultsAPI`, `ResultsService`, and serializers.
- **Notebook display helpers** for showing intermediate tables and plots.

*Outcome:* after this cell, every later step can focus on workflow logic rather than repeated setup code.

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
import re
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase.geometry.io import GeometryAPI
from owi.metadatabase.locations.io import LocationsAPI

from owi.metadatabase.results import ResultsAPI, ResultsService, WindSpeedHistogram
from owi.metadatabase.results.models import AnalysisDefinition
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer
from owi.metadatabase.results.services import ApiResultsRepository
from owi.metadatabase.results.utils import load_token_from_env_file


## Constants

This section defines the *runtime configuration* for the notebook.

**Review these values before execution**

- **`WORKBOOK`**: source Excel file containing the raw results.
- **`BASE_URL`**: results API root used for all backend calls.
- **`PROJECTSITE`** and **`MODEL_DEFINITION`**: metadata context used to resolve the correct backend objects.
- **`TOKEN`**: authentication token loaded from the environment file.
- **`ANALYSIS_ID`**: optional existing analysis identifier used to *skip upload* and reuse persisted backend data.

*Outcome:* the notebook has a single, explicit place where environment-specific values can be checked or changed.

In [2]:
WORKSPACE_ROOT = Path.cwd().resolve().parent
DEFAULT_WORKBOOK = WORKSPACE_ROOT / "scripts" / "data" / "results-example-data.xlsx"
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
DEFAULT_RESULTS_API_ROOT = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
BASE_URL = DEFAULT_RESULTS_API_ROOT
WORKBOOK = DEFAULT_WORKBOOK
PROJECTSITE = "Belwind"
MODEL_DEFINITION = f"as-designed {PROJECTSITE}"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE, TOKEN_ENV_VAR)
ANALYSIS_ID: int | None = None

display(pd.DataFrame([{"workbook": str(WORKBOOK),
                       "projectsite": PROJECTSITE,
                       "api_root": BASE_URL,
                       "token_available": TOKEN is not None,
                       "analysis_id": ANALYSIS_ID}]).T)
if TOKEN is None:
    raise ValueError(f"Set {TOKEN_ENV_VAR} in {DEFAULT_ENV_FILE} or export "
                     "it in the environment before running this notebook.")


,0
workbook,/home/pietro.dantuono@24SEA.local/Projects/SMA...
projectsite,Belwind
api_root,https://owimetadatabase-dev.azurewebsites.net/...
token_available,True
analysis_id,None


## Project and Location Setup

This step resolves the *backend metadata* required to attach workbook rows to the correct project and model definition.

**What happens here**

- Query the project site to obtain the backend **site id**.
- Query the geometry service to obtain the **model definition id**.
- Load asset locations for the selected project site.
- Build a lookup table from turbine title to backend location identifier.

*Why this matters:* the upload payloads must reference valid backend identifiers, otherwise the results cannot be associated with the correct project context.

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["Constants<br/>BASE_URL, PROJECTSITE, MODEL_DEFINITION"] --> B["LocationsAPI<br/>project site lookup"]
    A --> C["GeometryAPI<br/>model definition lookup"]
    B --> D["site_id"]
    B --> E["assetlocations"]
    C --> F["model_definition_id"]
    E --> G["location_frame"]
    G --> H["location_title_to_id_map"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef ids fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;
    classDef tables fill:#F8ECD9,stroke:#C08B3E,color:#50310E;

    class B,C api;
    class D,F,H ids;
    class E,G tables;
```

In [3]:
locations_api = LocationsAPI(api_root=BASE_URL, token=TOKEN)
geometry_api = GeometryAPI(api_root=BASE_URL, token=TOKEN)
site_id = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)["id"]
model_definition_id = geometry_api.get_modeldefinition_id(
    projectsite=PROJECTSITE, model_definition=MODEL_DEFINITION
)["id"]
assetlocations = locations_api.get_assetlocations(projectsite=PROJECTSITE)["data"]
location_frame = assetlocations.loc[
    :, [column for column in ["id", "title", "northing", "easting"]
        if column in assetlocations.columns]
].copy()
location_title_to_id_map = {str(row["title"]): int(row["id"])
                            for row in location_frame.to_dict(orient="records")
                            if row.get("title") is not None and row.get("id") is not None}
display(pd.DataFrame([{"site": PROJECTSITE,
                       "site_id": site_id,
                       "model_definition": MODEL_DEFINITION,
                       "model_definition_id": model_definition_id,
                       "num_locations": len(location_title_to_id_map)}]).T
                        .reset_index().rename(columns={"index": "metric", 0: "value"})
                        .set_index("metric"))


,value
metric,
site,Belwind
site_id,35
model_definition,as-designed Belwind
model_definition_id,4
num_locations,55


## Workbook Data Import

This section converts the Excel sheet into the *structured result objects* expected by the SDK and backend API.

**Key rules in this step**

- The sheet `Lifetime - Wind Histogram` is treated as the source of truth (header row at index 1).
- **`Title`**, **`Scope`**, and bin columns (e.g. `[0,1[`) are required.
- Any column whose name starts with `[` is treated as a histogram bin and its edges are parsed from the label.
- Scope values are mapped to backend location identifiers where a matching turbine title exists; site-level rows keep `location_id` as `None`.

**Transformation flow**

- Create an **`AnalysisDefinition`** describing the uploaded dataset.
- Convert workbook rows into `prepared_series` with site and location metadata attached.
- Validate and normalize those rows through `WindSpeedHistogram.to_results(...)`.
- Serialize the resulting objects into backend-ready payloads with `DjangoResultSerializer`.

*Outcome:* the notebook moves from raw spreadsheet rows to validated `result_payloads` that are ready for upload.

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["Workbook sheet<br/>Lifetime - Wind Histogram (header=1)"] --> B["pandas.read_excel"]
    B --> C["histogram_frame"]
    C --> D["bin_columns<br/>detect any column starting with ["]
    C --> E["prepared_series<br/>resolve scope, site, location, bins, values"]
    D --> E
    E --> F["WindSpeedHistogram.to_results"]
    F --> G["result_series"]
    G --> H["DjangoResultSerializer.to_payload"]
    H --> I["result_payloads"]

    classDef source fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B,C source;
    class D,E,F,H transform;
    class G,I output;
```

In [4]:
analysis = WindSpeedHistogram()
analysis_serializer = DjangoAnalysisSerializer()
result_serializer = DjangoResultSerializer()


def _parse_bin_edges(label: str) -> tuple[float, float]:
    """Extract the left and right edges from a histogram bin label like '[0,1['."""
    numbers = [float(value) for value in re.findall(r"-?\d+(?:\.\d+)?", label)]
    if len(numbers) < 2:
        raise ValueError(f"Could not parse histogram bin edges from {label!r}.")
    return numbers[0], numbers[1]


# -- Actual data processing and upload logic starts here --
sheet_name = "Lifetime - Wind Histogram"
histogram_frame = pd.read_excel(WORKBOOK, sheet_name=sheet_name, header=1)
bin_columns = [column for column in histogram_frame.columns
               if isinstance(column, str) and column.startswith("[")]
print("\033[95mInput data (histogram frame) loaded from Excel:\033[0m")
display(histogram_frame.head())
display(pd.DataFrame({"bin_column": bin_columns[:10],
                       "parsed_edges": [_parse_bin_edges(c) for c in bin_columns[:10]]}))

# Analysis payload creation requires an `AnalysisDefinition` and
# `AnalysisSerializer` configuration.
analysis_definition = AnalysisDefinition(
    name=analysis.analysis_name,
    model_definition_id=model_definition_id,
    location_id=None,
    source_type="notebook",
    source=str(WORKBOOK),
    description="Workbook upload for the wind speed histogram sheet.",
    additional_data={"sheet_name": sheet_name},
)
analysis_payload = analysis_serializer.to_payload(analysis_definition)

# `result_payloads` are created by first preparing the raw series from the input DataFrame
# into the structured format expected by the `ResultSeries` model, and then serialized
# into the format expected by the backend API through the `ResultSerializer`.
prepared_series = [{"title": str(row["Title"]),
                    "description": row.get("Description"),
                    "scope_label": str(row["Scope"]).strip(),
                    "site_id": site_id,
                    "location_id": (None if str(row["Scope"]).strip() == "Site"
                                    else location_title_to_id_map.get(str(row["Scope"]).strip())),
                    "bins": [_parse_bin_edges(str(column)) for column in bin_columns],
                    "values": [float(row[column]) for column in bin_columns],
                    "metadata": {"sheet_scope": str(row["Scope"]).strip()}}
                   for row in histogram_frame.to_dict(orient="records")]
# analysis.to_results performs the validation step of the prepared series
# against the result model, and converts them into a list of result
# objects. This step is crucial to ensure that the data conforms to the
# expected structure and types before serialization and upload.
result_series = analysis.to_results({"series": prepared_series})
result_payloads = [result_serializer.to_payload(item, analysis_id=0)
                   for item in result_series]

print("\033[95mPrepared result payloads ready for upload (showing first 5):\033[0m")
display(pd.DataFrame(result_payloads[:5]))
print("\033[95mLocation frame loaded from API (showing first 5):\033[0m")
display(location_frame.head())


Input data (histogram frame) loaded from Excel:


,Title,Description,Scope,"[0,1[","[1,2[","[2,3[","[3,4[","[4,5[","[5,6[","[6,7[",...,"[40,41[","[41,42[","[42,43[","[43,44[","[44,45[","[45,46[","[46,47[","[47,48[","[48,49[","[49,50["
0,Design,As-designed windspeed distribution as specifie...,Site,1,2,3,4,5.0,6.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,WTG1 - Y1,As measured distribution for location WTG1 - Y1,WTG1,50,49,48,47,46.0,45.0,44.0,...,10.0,9.0,8.0,7.0,6.0,5.0,4.0,3.0,2.0,1
2,WTG2 - Y1,As measured distribution for location WTG2 - Y1,WTG2,1,1,2,3,3.5,4.2,4.9,...,28.7,29.4,30.1,30.8,31.5,32.2,32.9,33.6,34.3,35
3,WTG1 - Y2,As measured distribution for location WTG1 - Y1,WTG1,50,49,48,47,46.0,45.0,44.0,...,10.0,9.0,8.0,7.0,6.0,5.0,4.0,3.0,2.0,1
4,WTG2 - Y2,As measured distribution for location WTG2 - Y1,WTG2,1,1,2,3,3.5,4.2,4.9,...,28.7,29.4,30.1,30.8,31.5,32.2,32.9,33.6,34.3,35


,bin_column,parsed_edges
0,"[0,1[","(0.0, 1.0)"
1,"[1,2[","(1.0, 2.0)"
2,"[2,3[","(2.0, 3.0)"
3,"[3,4[","(3.0, 4.0)"
4,"[4,5[","(4.0, 5.0)"
5,"[5,6[","(5.0, 6.0)"
6,"[6,7[","(6.0, 7.0)"
7,"[7,8[","(7.0, 8.0)"
8,"[8,9[","(8.0, 9.0)"
9,"[9,10[","(9.0, 10.0)"


Prepared result payloads ready for upload (showing first 5):


,analysis,site,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,short_description,description,additional_data
0,0,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",Design,As-designed windspeed distribution as specifie...,"{'analysis_kind': 'histogram', 'result_scope':..."
1,0,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG1 - Y1,As measured distribution for location WTG1 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."
2,0,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG2 - Y1,As measured distribution for location WTG2 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."
3,0,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG1 - Y2,As measured distribution for location WTG1 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."
4,0,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG2 - Y2,As measured distribution for location WTG2 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."


Location frame loaded from API (showing first 5):


,id,title,northing,easting
0,435,BBA01,51.697000,2.805556
1,436,BBA02,51.691333,2.802167
2,437,BBA03,51.685701,2.798815
3,438,BBA04,51.679944,2.795361
4,439,BBA05,51.674258,2.791977


## Upload Results via POST

This step creates backend records *only when needed*. If an existing analysis id is provided, the notebook reuses it and skips the upload.

**Execution logic**

- When **`ANALYSIS_ID` is `None`**:
  - create the analysis row first
  - attach the new analysis id to each result payload
  - upload the prepared results in bulk
- When **`ANALYSIS_ID` is already set**:
  - skip POST operations
  - continue with retrieval and plotting using the existing backend analysis

*Outcome:* the notebook supports both a full upload workflow and a faster read-only replay workflow.

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
flowchart TD
    A{"ANALYSIS_ID provided?"} -->|Yes| B["Reuse existing analysis_id"]
    A -->|No| C["create_analysis(analysis_payload)"]
    C --> D["analysis_id"]
    D --> E["Attach analysis id to each payload"]
    E --> F["create_results_bulk(upload_payloads)"]
    B --> G["Skip POST step"]
    F --> H["upload_log"]
    G --> H

    classDef decision fill:#FDEBD3,stroke:#C47A2C,color:#5A3412;
    classDef action fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef result fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A decision;
    class C,E,F,G action;
    class B,D,H result;
```

In [5]:
api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
analysis_id = ANALYSIS_ID
upload_payloads = []
create_results_response = {"exists": None}
if analysis_id is None:
    upload_payloads = list(result_payloads)
    created_analysis = api.create_analysis(analysis_payload)
    analysis_id = int(created_analysis["id"])

    create_results_response = api.create_results_bulk(
        [{**payload, "analysis": analysis_id} for payload in upload_payloads]
    )

upload_log = pd.DataFrame([{"analysis_id": analysis_id,
                            "uploaded_rows": len(upload_payloads),
                            "bulk_response_exists": create_results_response["exists"]}])

display(upload_log.T.reset_index().rename(columns={"index": "metric", 0: "value"}).set_index("metric"))


,value
metric,
analysis_id,55
uploaded_rows,5
bulk_response_exists,True


## Retrieve Persisted Results

This stage reads the saved backend analysis back into the notebook and reconstructs the normalized analysis frame.

**What this step does**

- Query the backend for the matching analysis definition.
- Use the returned analysis id to fetch persisted result rows.
- Deserialize each backend row into typed SDK result objects.
- Rebuild the normalized analysis DataFrame through `WindSpeedHistogram.from_results(...)`.

*Why this matters:* this confirms that the data can make a full round-trip from workbook to API and back into the SDK model layer.

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontFamily': 'IBM Plex Sans, Segoe UI, sans-serif',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["ResultsAPI.get_analysis<br/>name + model_definition + timestamp window"] --> B["retrieved_analysis"]
    B --> C["analysis_id"]
    C --> D["ResultsAPI.list_results<br/>analysis id filter"]
    D --> E["raw_results_frame"]
    E --> F["DjangoResultSerializer.from_mapping"]
    F --> G["retrieved_series"]
    G --> H["WindSpeedHistogram.from_results"]
    H --> I["retrieved_frame"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef objects fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef frames fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,D api;
    class B,C,F,G,H objects;
    class E,I frames;
```

In [6]:
api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
# Use the analysis_id obtained during the upload step;
# when running in skip-upload mode (ANALYSIS_ID preset),
# the variable was set in the upload cell directly.
raw_results_frame = api.list_results(analysis=analysis_id)["data"]
print("\033[95mRaw results frame retrieved from API:\033[0m")
display(raw_results_frame.head())
retrieved_series = [result_serializer.from_mapping(row)
                    for row in raw_results_frame.to_dict(orient="records")]
retrieved_frame = analysis.from_results(retrieved_series)
print("\033[95mRetrieved results frame reconstructed from API data:\033[0m")
display(retrieved_frame.head())


Raw results frame retrieved from API:


,id,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,analysis,site,location,short_description,description,related_object,additional_data,slug,visibility,visibility_groups
0,4578,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",55,35,None,Design,As-designed windspeed distribution as specifie...,None,"{'series_key': 'Design', 'scope_label': 'Site'...",belwind_none_windspeedhistogram_55_design_4578,usergroup,[]
1,4579,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",55,35,None,WTG1 - Y1,As measured distribution for location WTG1 - Y1,None,"{'series_key': 'WTG1 - Y1', 'scope_label': 'WT...",belwind_none_windspeedhistogram_55_wtg1-y1_4579,usergroup,[]
2,4581,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",55,35,None,WTG1 - Y2,As measured distribution for location WTG1 - Y1,None,"{'series_key': 'WTG1 - Y2', 'scope_label': 'WT...",belwind_none_windspeedhistogram_55_wtg1-y2_4581,usergroup,[]
3,4580,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",55,35,None,WTG2 - Y1,As measured distribution for location WTG2 - Y1,None,"{'series_key': 'WTG2 - Y1', 'scope_label': 'WT...",belwind_none_windspeedhistogram_55_wtg2-y1_4580,usergroup,[]
4,4582,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",55,35,None,WTG2 - Y2,As measured distribution for location WTG2 - Y1,None,"{'series_key': 'WTG2 - Y2', 'scope_label': 'WT...",belwind_none_windspeedhistogram_55_wtg2-y2_4582,usergroup,[]


Retrieved results frame reconstructed from API data:


,series_name,scope,bin_left,bin_right,value
0,Design,Site,0.0,1.0,1.0
1,Design,Site,1.0,2.0,2.0
2,Design,Site,2.0,3.0,3.0
3,Design,Site,3.0,4.0,4.0
4,Design,Site,4.0,5.0,5.0


## Plot Results

The final stage renders the retrieved backend data through `ResultsService` so the same analysis can be inspected visually.

**The notebook generates one view**

- **Histogram plot**: shows the wind speed distribution as a bar chart with bin edges on the x-axis and counts on the y-axis.

**What to look for**

- Whether the retrieved dataset has the expected number of rows and series.
- Whether the histogram view is produced successfully.
- Whether the plotted values are consistent with the workbook content and uploaded backend data.

*Outcome:* this step confirms that the uploaded analysis is not only stored correctly, but also consumable through the plotting layer exposed by the SDK.

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["ApiResultsRepository(api)"] --> B["ResultsService"]
    C["filters<br/>analysis_id"] --> B
    B --> D["plot_results(..., plot_type=histogram)"]
    D --> E["histogram_plot.notebook"]
    E --> F["display(...)"]

    classDef service fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef request fill:#FDEBD3,stroke:#C47A2C,color:#5A3412;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B service;
    class C,D request;
    class E,F output;
```

In [7]:
results_service = ResultsService(repository=ApiResultsRepository(api=api))
filters = {"analysis_id": analysis_id}

histogram_plot = results_service.plot_results(
    analysis.analysis_name,
    filters=filters,
    plot_type="histogram",
)

display(histogram_plot.notebook)


In [8]:
plot_summary = pd.DataFrame(
    [
        {
            "analysis_id": analysis_id,
            "retrieved_rows": len(raw_results_frame),
            "normalized_rows": len(retrieved_frame),
            "series_count": len(retrieved_series),
            "histogram_plot_available": histogram_plot.notebook is not None,
        }
    ]
).T.reset_index().rename(columns={"index": "metric", 0: "value"}).set_index("metric")

display(plot_summary)


,value
metric,
analysis_id,55
retrieved_rows,5
normalized_rows,250
series_count,5
histogram_plot_available,True
